# 11_Scatterplot_linear_regression_by_threshold

This notebook runs a per-sample linear regression comparing Mutect2 vs DeepVariant values from the merged exact-variant CSVs.

For each sample and threshold setting, it fits:

`DeepVariant value = slope × Mutect2 value + intercept`

The main outputs are:

1. A CSV table with slope, intercept, R², p-value, and number of variants for each sample/threshold.
2. Slope-by-sample plots.
3. R²-by-sample plots.
4. Histograms showing the distribution of slope and R² values across samples.

By default, this notebook runs the regression on the VAF scatterplots. You can add `alt_depth` or `total_depth` to `metrics_to_regress` if you want those too.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

In [ ]:
base_dir = Path(r"/project/knathans_shared/donetski/Notebooks/OutputFiles/09_merged_HGVSg_HGVSp_by_exact_variant")
base_output_dir= Path(r"/project/knathans_shared/donetski/Notebooks/OutputFiles")
output_dir = base_dir / "11_scatterplot_regression_by_threshold"
output_dir.mkdir(parents=True, exist_ok=True)

# Use None to process every sample, or an integer like 5 for testing.
sample_limit = None

# Same threshold settings used in Script 10.
vaf_thresholds = [0.01, 0.03, 0.05, 0.07, 0.10]
altdepth_thresholds = [1, 3, 5, 7, 10]

include_no_threshold = True
include_independent_thresholds = True
include_combo_thresholds = True

# Start with VAF regression because that is the main comparison.
# Add "alt_depth" and/or "total_depth" later if you want regression summaries for those scatterplots too.
metrics_to_regress = ["vaf"]
# metrics_to_regress = ["vaf", "alt_depth", "total_depth"]

show_plots = False


## Find input CSVs

Each sample should have one `unique_exact_variants.csv` file inside its sample folder.


In [ ]:
sample_files = sorted(base_dir.glob("*/*_combined_HGVSg_HGVSp_unique_exact_variants.csv"))

if sample_limit is not None:
    sample_files = sample_files[:sample_limit]

print("Sample files found:", len(sample_files))
for path in sample_files[:5]:
    print(path)

## Regression settings

These columns match the merged Mutect2/DeepVariant comparison CSVs.


In [ ]:
metric_info = {
    "vaf": {
        "x_col": "Mutect2_VAF",
        "y_col": "DeepVariant_VAF",
        "x_label": "Mutect2 VAF",
        "y_label": "DeepVariant VAF",
        "expected_slope": 1,
    },
    "alt_depth": {
        "x_col": "Mutect2_alt_depth",
        "y_col": "DeepVariant_alt_depth",
        "x_label": "Mutect2 alternate depth",
        "y_label": "DeepVariant alternate depth",
        "expected_slope": 1,
    },
    "total_depth": {
        "x_col": "Mutect2_total_depth",
        "y_col": "DeepVariant_total_depth",
        "x_label": "Mutect2 total depth",
        "y_label": "DeepVariant total depth",
        "expected_slope": 1,
    },
}

required_columns = [
    "Mutect2_found",
    "DeepVariant_found",
    "Mutect2_VAF",
    "DeepVariant_VAF",
    "Mutect2_alt_depth",
    "DeepVariant_alt_depth",
    "Mutect2_total_depth",
    "DeepVariant_total_depth",
]

## Helper functions

The threshold filters are applied to rows found by both callers.

For VAF thresholds, both callers must pass the VAF cutoff.

For AltDepth thresholds, both callers must pass the AltDepth cutoff.


In [ ]:
def is_found(series):
    return series.astype(str).str.upper().eq("Y")


def format_threshold(value):
    return str(value).replace(".", "p")


def make_threshold_configs():
    configs = []

    if include_no_threshold:
        configs.append({
            "threshold_type": "none",
            "threshold_label": "no_threshold",
            "vaf_threshold": None,
            "altdepth_threshold": None,
        })

    if include_independent_thresholds:
        for threshold in vaf_thresholds:
            configs.append({
                "threshold_type": "vaf_only",
                "threshold_label": f"vaf_ge_{format_threshold(threshold)}",
                "vaf_threshold": threshold,
                "altdepth_threshold": None,
            })

        for threshold in altdepth_thresholds:
            configs.append({
                "threshold_type": "altdepth_only",
                "threshold_label": f"altdepth_ge_{threshold}",
                "vaf_threshold": None,
                "altdepth_threshold": threshold,
            })

    if include_combo_thresholds:
        for altdepth_threshold in altdepth_thresholds:
            for vaf_threshold in vaf_thresholds:
                configs.append({
                    "threshold_type": "combo",
                    "threshold_label": f"altdepth_ge_{altdepth_threshold}_vaf_ge_{format_threshold(vaf_threshold)}",
                    "vaf_threshold": vaf_threshold,
                    "altdepth_threshold": altdepth_threshold,
                })

    return configs


def apply_thresholds(df, threshold_config):
    out_df = df.copy()

    vaf_threshold = threshold_config["vaf_threshold"]
    altdepth_threshold = threshold_config["altdepth_threshold"]

    if vaf_threshold is not None:
        out_df = out_df[
            (out_df["Mutect2_VAF"] >= vaf_threshold) &
            (out_df["DeepVariant_VAF"] >= vaf_threshold)
        ].copy()

    if altdepth_threshold is not None:
        out_df = out_df[
            (out_df["Mutect2_alt_depth"] >= altdepth_threshold) &
            (out_df["DeepVariant_alt_depth"] >= altdepth_threshold)
        ].copy()

    return out_df


def run_regression(plot_df, x_col, y_col):
    plot_df = plot_df[[x_col, y_col]].copy()
    plot_df[x_col] = pd.to_numeric(plot_df[x_col], errors="coerce")
    plot_df[y_col] = pd.to_numeric(plot_df[y_col], errors="coerce")
    plot_df = plot_df.dropna(subset=[x_col, y_col])

    n_points = len(plot_df)

    empty_result = {
        "n_points": n_points,
        "slope": np.nan,
        "intercept": np.nan,
        "r_value": np.nan,
        "r_squared": np.nan,
        "p_value": np.nan,
        "slope_stderr": np.nan,
    }

    if n_points < 3:
        empty_result["regression_status"] = "too_few_points"
        return empty_result

    if plot_df[x_col].nunique(dropna=True) < 2:
        empty_result["regression_status"] = "constant_x"
        return empty_result

    result = linregress(plot_df[x_col], plot_df[y_col])

    return {
        "regression_status": "ok",
        "n_points": n_points,
        "slope": result.slope,
        "intercept": result.intercept,
        "r_value": result.rvalue,
        "r_squared": result.rvalue ** 2,
        "p_value": result.pvalue,
        "slope_stderr": result.stderr,
    }


def make_regression_equation(regression, x_col, y_col):
    if regression["regression_status"] != "ok":
        return np.nan

    return f"{y_col} = {regression['slope']:.6f} * {x_col} + {regression['intercept']:.6f}"


## Run linear regressions

This computes one regression per sample, per threshold, per selected metric.


In [ ]:
summary_rows = []

for sample_file in sample_files:
    sample_id = sample_file.parent.name
    df = pd.read_csv(sample_file, low_memory=False)

    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        print(f"Skipping {sample_id}; missing columns: {missing}")
        continue

    for column in required_columns[2:]:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    mutect2_found = is_found(df["Mutect2_found"])
    deepvariant_found = is_found(df["DeepVariant_found"])
    both_found = mutect2_found & deepvariant_found
    both_df = df.loc[both_found].copy()

    for threshold_config in make_threshold_configs():
        threshold_df = apply_thresholds(both_df, threshold_config)

        for metric in metrics_to_regress:
            info = metric_info[metric]
            regression = run_regression(threshold_df, info["x_col"], info["y_col"])
            regression_equation = make_regression_equation(regression, info["x_col"], info["y_col"])

            summary_rows.append({
                "sample_id": sample_id,
                "metric": metric,
                "x_col": info["x_col"],
                "y_col": info["y_col"],
                "threshold_type": threshold_config["threshold_type"],
                "threshold_label": threshold_config["threshold_label"],
                "vaf_threshold": threshold_config["vaf_threshold"],
                "altdepth_threshold": threshold_config["altdepth_threshold"],
                "total_unique_variants": len(df),
                "mutect2_found": int(mutect2_found.sum()),
                "deepvariant_found": int(deepvariant_found.sum()),
                "both_callers_found": int(both_found.sum()),
                "variants_after_threshold": len(threshold_df),
                "percent_both_callers_retained": len(threshold_df) / int(both_found.sum()) * 100 if int(both_found.sum()) > 0 else np.nan,
                "regression_equation": regression_equation,
                **regression,
            })

regression_df = pd.DataFrame(summary_rows)

summary_csv = output_dir / "11_scatterplot_regression_by_threshold_summary.csv"
regression_df.to_csv(summary_csv, index=False)

print("Saved:", summary_csv)
regression_df.head()


## Plot slope and R² summaries

There are two kinds of plots:

1. `*_by_sample`: x-axis is sample ID, useful for finding outlier samples.
2. `*_histogram`: x-axis is slope or R² value, useful for seeing the distribution across all samples.


In [ ]:
slope_by_sample_dir = output_dir / "slope_by_sample"
r2_by_sample_dir = output_dir / "r2_by_sample"
slope_hist_dir = output_dir / "slope_histograms"
r2_hist_dir = output_dir / "r2_histograms"

for directory in [slope_by_sample_dir, r2_by_sample_dir, slope_hist_dir, r2_hist_dir]:
    directory.mkdir(parents=True, exist_ok=True)


def safe_name(value):
    return str(value).replace(" ", "_").replace("/", "_").replace(".", "p")


def save_current_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    if show_plots:
        plt.show()
    else:
        plt.close()
    print("Saved:", path)


for (metric, threshold_label), plot_df in regression_df.groupby(["metric", "threshold_label"]):
    plot_df = plot_df[plot_df["regression_status"].eq("ok")].copy()
    plot_df = plot_df.sort_values("sample_id")

    if plot_df.empty:
        print(f"Skipping plots for {metric} {threshold_label}; no successful regressions")
        continue

    expected_slope = metric_info[metric]["expected_slope"]
    file_prefix = f"{safe_name(metric)}_{safe_name(threshold_label)}"

    # Slope by sample: x-axis is sample ID.
    plt.figure(figsize=(max(12, len(plot_df) * 0.12), 5))
    plt.scatter(plot_df["sample_id"], plot_df["slope"], s=18)
    plt.axhline(expected_slope, linestyle="--", linewidth=1, label="Expected slope = 1")
    plt.xlabel("Sample ID")
    plt.ylabel("Slope")
    plt.title(f"{metric}: slope by sample ({threshold_label})")
    plt.xticks(rotation=90, fontsize=5)
    plt.legend()
    save_current_plot(slope_by_sample_dir / f"{file_prefix}_slope_by_sample.png")

    # R² by sample: x-axis is sample ID.
    plt.figure(figsize=(max(12, len(plot_df) * 0.12), 5))
    plt.scatter(plot_df["sample_id"], plot_df["r_squared"], s=18)
    plt.axhline(1, linestyle="--", linewidth=1, label="R² = 1")
    plt.xlabel("Sample ID")
    plt.ylabel("R²")
    plt.ylim(-0.02, 1.02)
    plt.title(f"{metric}: R² by sample ({threshold_label})")
    plt.xticks(rotation=90, fontsize=5)
    plt.legend()
    save_current_plot(r2_by_sample_dir / f"{file_prefix}_r2_by_sample.png")

    # Slope histogram: x-axis is slope value.
    plt.figure(figsize=(8, 5))
    plt.hist(plot_df["slope"].dropna(), bins=30)
    plt.axvline(expected_slope, linestyle="--", linewidth=1, label="Expected slope = 1")
    plt.xlabel("Slope")
    plt.ylabel("Number of samples")
    plt.title(f"{metric}: slope distribution ({threshold_label})")
    plt.legend()
    save_current_plot(slope_hist_dir / f"{file_prefix}_slope_histogram.png")

    # R² histogram: x-axis is R² value.
    plt.figure(figsize=(8, 5))
    plt.hist(plot_df["r_squared"].dropna(), bins=30)
    plt.xlabel("R²")
    plt.ylabel("Number of samples")
    plt.xlim(0, 1)
    plt.title(f"{metric}: R² distribution ({threshold_label})")
    save_current_plot(r2_hist_dir / f"{file_prefix}_r2_histogram.png")

## Boxplots by threshold

These boxplots summarize the distribution of sample-level slopes and R² values across thresholds. The box shows Q1–Q3, the center line is the median, and the marker shows the mean.


In [ ]:
boxplot_dir = output_dir / "boxplots"
vaf_only_boxplot_dir = boxplot_dir / "vaf_only"
altdepth_only_boxplot_dir = boxplot_dir / "altdepth_only"
combo_fixed_altdepth_dir = boxplot_dir / "combo_fixed_altdepth"
combo_fixed_vaf_dir = boxplot_dir / "combo_fixed_vaf"

for directory in [vaf_only_boxplot_dir, altdepth_only_boxplot_dir, combo_fixed_altdepth_dir, combo_fixed_vaf_dir]:
    directory.mkdir(parents=True, exist_ok=True)


def get_boxplot_values(summary_df, metric, threshold_label, value_col):
    return summary_df[
        summary_df["metric"].eq(metric) &
        summary_df["threshold_label"].eq(threshold_label) &
        summary_df["regression_status"].eq("ok")
    ][value_col].dropna()


def save_boxplot(data, labels, title, ylabel, out_png, expected_line=None):
    if all(len(values) == 0 for values in data):
        print("Skipped empty boxplot:", out_png)
        return

    plt.figure(figsize=(max(8, len(labels) * 1.4), 5))
    plt.boxplot(data, labels=labels, showmeans=True)

    if expected_line is not None:
        plt.axhline(expected_line, linestyle="--", linewidth=1)

    plt.ylabel(ylabel)
    plt.title(title)
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)

    if show_plots:
        plt.show()
    else:
        plt.close()

    print("Saved:", out_png)


for metric in metrics_to_regress:
    expected_slope = metric_info[metric]["expected_slope"]

    # VAF-only thresholds on one figure.
    vaf_configs = [
        {
            "threshold_label": f"vaf_ge_{format_threshold(vaf_threshold)}",
            "label": f"VAF ≥ {vaf_threshold:g}",
        }
        for vaf_threshold in vaf_thresholds
    ]

    for value_col, ylabel, expected_line in [
        ("slope", "Slope", expected_slope),
        ("r_squared", "R²", 1),
    ]:
        data = [get_boxplot_values(regression_df, metric, config["threshold_label"], value_col) for config in vaf_configs]
        labels = [config["label"] for config in vaf_configs]
        save_boxplot(
            data,
            labels,
            f"{metric}: {ylabel} by VAF threshold",
            ylabel,
            vaf_only_boxplot_dir / f"{safe_name(metric)}_{safe_name(value_col)}_vaf_only_boxplot.png",
            expected_line=expected_line,
        )

    # AltDepth-only thresholds on one figure.
    altdepth_configs = [
        {
            "threshold_label": f"altdepth_ge_{altdepth_threshold}",
            "label": f"AltDepth ≥ {altdepth_threshold}",
        }
        for altdepth_threshold in altdepth_thresholds
    ]

    for value_col, ylabel, expected_line in [
        ("slope", "Slope", expected_slope),
        ("r_squared", "R²", 1),
    ]:
        data = [get_boxplot_values(regression_df, metric, config["threshold_label"], value_col) for config in altdepth_configs]
        labels = [config["label"] for config in altdepth_configs]
        save_boxplot(
            data,
            labels,
            f"{metric}: {ylabel} by AltDepth threshold",
            ylabel,
            altdepth_only_boxplot_dir / f"{safe_name(metric)}_{safe_name(value_col)}_altdepth_only_boxplot.png",
            expected_line=expected_line,
        )

    # Combo thresholds: for each AltDepth threshold, vary VAF along the x-axis.
    for altdepth_threshold in altdepth_thresholds:
        combo_configs = [
            {
                "threshold_label": f"altdepth_ge_{altdepth_threshold}_vaf_ge_{format_threshold(vaf_threshold)}",
                "label": f"VAF ≥ {vaf_threshold:g}",
            }
            for vaf_threshold in vaf_thresholds
        ]

        for value_col, ylabel, expected_line in [
            ("slope", "Slope", expected_slope),
            ("r_squared", "R²", 1),
        ]:
            data = [get_boxplot_values(regression_df, metric, config["threshold_label"], value_col) for config in combo_configs]
            labels = [config["label"] for config in combo_configs]
            save_boxplot(
                data,
                labels,
                f"{metric}: {ylabel}, AltDepth ≥ {altdepth_threshold} with VAF varied",
                ylabel,
                combo_fixed_altdepth_dir / f"{safe_name(metric)}_{safe_name(value_col)}_altdepth_ge_{altdepth_threshold}_vaf_varied_boxplot.png",
                expected_line=expected_line,
            )

    # Optional opposite view: for each VAF threshold, vary AltDepth along the x-axis.
    for vaf_threshold in vaf_thresholds:
        combo_configs = [
            {
                "threshold_label": f"altdepth_ge_{altdepth_threshold}_vaf_ge_{format_threshold(vaf_threshold)}",
                "label": f"AltDepth ≥ {altdepth_threshold}",
            }
            for altdepth_threshold in altdepth_thresholds
        ]

        for value_col, ylabel, expected_line in [
            ("slope", "Slope", expected_slope),
            ("r_squared", "R²", 1),
        ]:
            data = [get_boxplot_values(regression_df, metric, config["threshold_label"], value_col) for config in combo_configs]
            labels = [config["label"] for config in combo_configs]
            save_boxplot(
                data,
                labels,
                f"{metric}: {ylabel}, VAF ≥ {vaf_threshold:g} with AltDepth varied",
                ylabel,
                combo_fixed_vaf_dir / f"{safe_name(metric)}_{safe_name(value_col)}_vaf_ge_{format_threshold(vaf_threshold)}_altdepth_varied_boxplot.png",
                expected_line=expected_line,
            )


## Quick threshold comparison table

This summarizes the median slope, median R², and median variant retention for each threshold.


In [ ]:
ok_regression_df = regression_df[regression_df["regression_status"].eq("ok")].copy()

threshold_summary_df = (
    ok_regression_df
    .groupby(["metric", "threshold_type", "vaf_threshold", "altdepth_threshold", "threshold_label"], dropna=False)
    .agg(
        samples_with_successful_regression=("sample_id", "nunique"),
        median_n_points=("n_points", "median"),
        median_percent_retained=("percent_both_callers_retained", "median"),
        mean_percent_retained=("percent_both_callers_retained", "mean"),
        median_slope=("slope", "median"),
        mean_slope=("slope", "mean"),
        q1_slope=("slope", lambda x: x.quantile(0.25)),
        q3_slope=("slope", lambda x: x.quantile(0.75)),
        median_r_squared=("r_squared", "median"),
        mean_r_squared=("r_squared", "mean"),
        q1_r_squared=("r_squared", lambda x: x.quantile(0.25)),
        q3_r_squared=("r_squared", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)

threshold_summary_csv = output_dir / "11_scatterplot_regression_threshold_level_summary.csv"
threshold_summary_df.to_csv(threshold_summary_csv, index=False)

print("Saved:", threshold_summary_csv)
threshold_summary_df
